In [ ]:
# Example 1: HTML component construction with the FT (FastTags) DSL.
# to_xml() renders a component to HTML; str() of a component that has an id returns
# that id, for use in HTMX target references.
from fasthtml.common import Div, P, H1, A, Span, to_xml

page = Div(
    H1("Hello, FastHTML"),
    P("This is a paragraph with a ", A("link", href="https://example.com"), "."),
    Span("inline span", cls="highlight"),
    id="main",
)

rendered = to_xml(page)
print(rendered)
assert str(page) == "main"

assert "<div id=\"main\"" in rendered
assert "<h1>Hello, FastHTML</h1>" in rendered
assert "href=\"https://example.com\"" in rendered
assert "class=\"highlight\"" in rendered

In [ ]:
# Example 2: Build a FastHTML app, register a route, and verify the app object.
from fasthtml.common import FastHTML, Titled, P

# A fixed secret_key; without one FastHTML creates a key file (.sesskey) in the working directory
app = FastHTML(secret_key="example-secret-key")

@app.get("/")
def home():
    return Titled("Home", P("Welcome to FastHTML"))

@app.get("/about")
def about():
    return Titled("About", P("About page"))

# The app should expose the underlying Starlette router with our routes.
route_paths = {r.path for r in app.router.routes if hasattr(r, "path")}
print(f"Registered route paths: {sorted(route_paths)}")

assert "/" in route_paths
assert "/about" in route_paths

In [ ]:
# Example 3: End-to-end request handling using Starlette's TestClient.
from starlette.testclient import TestClient

from fasthtml.common import FastHTML, Titled, P

# A fixed secret_key; without one FastHTML creates a key file (.sesskey) in the working directory
app = FastHTML(secret_key="example-secret-key")

@app.get("/")
def index():
    return Titled("Index", P("hello world"))

client = TestClient(app)
response = client.get("/")
print(f"status_code = {response.status_code}")
print(f"body[:200]  = {response.text[:200]!r}")

assert response.status_code == 200
assert "hello world" in response.text
assert "<title>Index</title>" in response.text

In [ ]:
# Example 4: Path parameters and typed conversion through a FastHTML handler.
from starlette.testclient import TestClient

from fasthtml.common import FastHTML, P

# A fixed secret_key; without one FastHTML creates a key file (.sesskey) in the working directory
app = FastHTML(secret_key="example-secret-key")

@app.get("/square/{n}")
def square(n: int):
    return P(f"{n} squared is {n * n}")

client = TestClient(app)
r = client.get("/square/7")
print(f"status = {r.status_code}")
print(f"body   = {r.text!r}")

assert r.status_code == 200
assert "49" in r.text

In [ ]:
# Example 5: HTMX integration via the hx_* attribute helpers on FT tags.
from fasthtml.common import Button, Div, to_xml

btn = Button(
    "Click me",
    hx_get="/clicked",
    hx_target="#result",
    hx_swap="innerHTML",
)
result = Div(id="result")

btn_html = to_xml(btn)
result_html = to_xml(result)
print(btn_html)
print(result_html)

assert "hx-get=\"/clicked\"" in btn_html
assert "hx-target=\"#result\"" in btn_html
assert "hx-swap=\"innerHTML\"" in btn_html
assert "id=\"result\"" in result_html